In [1]:
import tkinter as tk
from tkinter import filedialog, messagebox
from PIL import Image, ImageTk
import cv2
import numpy as np
import tensorflow as tf
from datetime import datetime

# ==========================
# CONFIGURATION
# ==========================

MODEL_PATH = "asl_model.h5"
IMG_SIZE = 64

CLASSES = [
    'A','B','C','D','E','F','G','H','I','J',
    'K','L','M','N','O','P','Q','R','S','T',
    'U','V','W','X','Y','Z',
    'del','nothing','space'
]

# ==========================
# LOAD MODEL
# ==========================

model = tf.keras.models.load_model(MODEL_PATH)

# ==========================
# TIME RESTRICTION
# ==========================

def check_time():
    current_hour = datetime.now().hour
    return 18 <= current_hour < 22

# ==========================
# MAIN APP
# ==========================

class SignLanguageApp:

    def __init__(self, root):

        self.root = root
        self.root.title("Sign Language Detection System")
        self.root.geometry("1000x700")

        self.cap = None
        self.running = False

        title = tk.Label(
            root,
            text="Sign Language Detection System",
            font=("Arial", 22, "bold")
        )
        title.pack(pady=10)

        self.video_label = tk.Label(root)
        self.video_label.pack()

        self.result_label = tk.Label(
            root,
            text="Prediction: None",
            font=("Arial", 18)
        )
        self.result_label.pack(pady=10)

        self.conf_label = tk.Label(
            root,
            text="Confidence: 0%",
            font=("Arial", 16)
        )
        self.conf_label.pack()

        btn_frame = tk.Frame(root)
        btn_frame.pack(pady=20)

        upload_btn = tk.Button(
            btn_frame,
            text="Upload Image",
            font=("Arial", 14),
            command=self.upload_image,
            width=15
        )
        upload_btn.grid(row=0, column=0, padx=10)

        webcam_btn = tk.Button(
            btn_frame,
            text="Start Webcam",
            font=("Arial", 14),
            command=self.start_webcam,
            width=15
        )
        webcam_btn.grid(row=0, column=1, padx=10)

        stop_btn = tk.Button(
            btn_frame,
            text="Stop Webcam",
            font=("Arial", 14),
            command=self.stop_webcam,
            width=15
        )
        stop_btn.grid(row=0, column=2, padx=10)

    # ==========================
    # IMAGE PREPROCESSING
    # ==========================

    def preprocess(self, image):

        image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
        image = image.astype("float32") / 255.0
        image = np.expand_dims(image, axis=0)

        return image

    # ==========================
    # PREDICTION
    # ==========================

    def predict(self, image):

        processed = self.preprocess(image)

        pred = model.predict(processed, verbose=0)

        idx = np.argmax(pred)

        confidence = np.max(pred)

        label = CLASSES[idx]

        return label, confidence

    # ==========================
    # UPLOAD IMAGE
    # ==========================

    def upload_image(self):

        if not check_time():
            messagebox.showwarning(
                "Restricted",
                "System accessible only between 6 PM and 10 PM"
            )
            return

        file_path = filedialog.askopenfilename()

        if not file_path:
            return

        image = cv2.imread(file_path)

        if image is None:
            messagebox.showerror(
                "Error",
                "Unable to open image."
            )
            return

        label, confidence = self.predict(image)

        self.result_label.config(
            text=f"Prediction: {label}"
        )

        self.conf_label.config(
            text=f"Confidence: {confidence*100:.2f}%"
        )

        image_rgb = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        image_rgb = cv2.resize(
            image_rgb,
            (400,400)
        )

        img = ImageTk.PhotoImage(
            Image.fromarray(image_rgb)
        )

        self.video_label.config(image=img)
        self.video_label.image = img

    # ==========================
    # START WEBCAM
    # ==========================

    def start_webcam(self):

        if not check_time():
            messagebox.showwarning(
                "Restricted",
                "System accessible only between 6 PM and 10 PM"
            )
            return

        self.cap = cv2.VideoCapture(0)

        self.running = True

        self.update_frame()

    # ==========================
    # UPDATE VIDEO
    # ==========================

    def update_frame(self):

        if not self.running:
            return

        ret, frame = self.cap.read()

        if ret:

            label, confidence = self.predict(frame)

            cv2.putText(
                frame,
                f"{label} ({confidence*100:.1f}%)",
                (20,40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0,255,0),
                2
            )

            frame_rgb = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )

            img = ImageTk.PhotoImage(
                Image.fromarray(frame_rgb)
            )

            self.video_label.config(image=img)
            self.video_label.image = img

            self.result_label.config(
                text=f"Prediction: {label}"
            )

            self.conf_label.config(
                text=f"Confidence: {confidence*100:.2f}%"
            )

        self.root.after(10, self.update_frame)

    # ==========================
    # STOP WEBCAM
    # ==========================

    def stop_webcam(self):

        self.running = False

        if self.cap is not None:
            self.cap.release()

# ==========================
# RUN APP
# ==========================

root = tk.Tk()

app = SignLanguageApp(root)

root.mainloop()

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'asl_model.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)